# Minimal LangGraph Agent for SageMaker Studio

A simple LangGraph agent using AWS Bedrock Claude.

## Setup (Run from CLI before using this notebook)

```bash
# Create an inference profile (run ONE of these)
./setup-inference-profile.sh haiku      # Fast & cheap - great for testing
./setup-inference-profile.sh sonnet     # Balanced - recommended
./setup-inference-profile.sh --test haiku  # Create and test

# See all options
./setup-inference-profile.sh --help
```

## Configuration (Next Cell)

1. Set `MODEL` to match what you ran (e.g., `"haiku"` or `"sonnet"`)
2. Paste the ARN output into `INFERENCE_PROFILE_ARN`

In [ ]:
#################################################
# CONFIGURATION
# 1. Set MODEL to match what you used with setup-inference-profile.sh
# 2. Paste your inference profile ARN
#################################################

MODEL = "haiku"  # Options: haiku, sonnet, haiku45, sonnet45
INFERENCE_PROFILE_ARN = "PASTE_YOUR_ARN_HERE"  # <-- PASTE HERE
REGION = "us-west-2"

#################################################

# Model ID mapping (used to bypass bedrock:GetInferenceProfile permission)
BASE_MODEL_IDS = {
    "haiku": "anthropic.claude-3-5-haiku-20241022-v1:0",
    "sonnet": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "haiku45": "anthropic.claude-haiku-4-5-20251001-v1:0",
    "sonnet45": "anthropic.claude-sonnet-4-5-20250929-v1:0",
}

# Validate configuration
if "PASTE" in INFERENCE_PROFILE_ARN or "YOUR" in INFERENCE_PROFILE_ARN:
    print("ERROR: Please paste your inference profile ARN above!")
    print("")
    print("Run this from CLI first:")
    print("  ./setup-inference-profile.sh haiku")
    print("")
    print("Then copy the ARN output and paste it above.")
elif MODEL not in BASE_MODEL_IDS:
    print(f"ERROR: Unknown MODEL '{MODEL}'")
    print(f"Valid options: {list(BASE_MODEL_IDS.keys())}")
else:
    BASE_MODEL_ID = BASE_MODEL_IDS[MODEL]
    print(f"Model:   {MODEL} ({BASE_MODEL_ID})")
    print(f"Profile: {INFERENCE_PROFILE_ARN}")
    print(f"Region:  {REGION}")
    print("Configuration OK!")

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "langchain-mcp-adapters",
    "mcp",
    "httpx",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install missing packages
%pip install langgraph>=1.0.6 -q

## 2. Imports

In [ ]:
from datetime import datetime
from typing import Literal

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

print("Imports successful!")

## 3. Define Tools

Simple tools for testing the agent's tool-calling capabilities.

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


tools = [get_current_time, add_numbers]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## 4. Initialize LLM

Uses the configuration from the first cell. The `base_model_id` parameter tells langchain-aws which underlying model is used, bypassing the `bedrock:GetInferenceProfile` API call that SageMaker Unified Studio roles don't have permission for.

In [ ]:
# Uses INFERENCE_PROFILE_ARN, BASE_MODEL_ID, and REGION from the configuration cell

llm = ChatBedrockConverse(
    model=INFERENCE_PROFILE_ARN,
    provider="anthropic",  # Required when using ARN
    region_name=REGION,
    temperature=0,
    base_model_id=BASE_MODEL_ID,  # Bypasses bedrock:GetInferenceProfile permission requirement
)

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

print("LLM initialized!")
print(f"Model:   {MODEL} ({BASE_MODEL_ID})")
print(f"Profile: {INFERENCE_PROFILE_ARN}")
print(f"Region:  {REGION}")

## 5. Build the LangGraph Agent

A minimal ReAct-style agent with:
- `agent` node: calls the LLM
- `tools` node: executes tools
- Conditional edge: routes to tools or ends

In [ ]:
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Determine whether to continue to tools or end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


def call_model(state: MessagesState):
    """Call the LLM."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# Build the graph
graph = StateGraph(MessagesState)

# Add nodes
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(tools))

# Add edges
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")

# Compile
agent = graph.compile()

print("Agent graph compiled successfully!")

## 6. Run the Agent

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"Question: {question}")
    print("-" * 50)

    result = agent.invoke(
        {
            "messages": [
                SystemMessage(content="You are a helpful assistant. Use tools when needed."),
                HumanMessage(content=question),
            ]
        }
    )

    final_message = result["messages"][-1]
    print(f"\nResponse:\n{final_message.content}")
    return result

In [ ]:
# Test: Get current time
result = run_agent("What is the current time?")

In [ ]:
# Test: Math calculation
result = run_agent("What is 42 + 17?")

In [ ]:
# Test: Multiple tools
result = run_agent("What time is it and what is 100 + 200?")

## 7. Try Your Own Question

In [ ]:
# Enter your own question here
my_question = "Add 999 and 1, then tell me the time."

result = run_agent(my_question)